# Byzantine FL on Bank Transaction Data

This notebook adapts the `byzantine-fl/transaction_byzantine_demo.ipynb` experiment flow to `bank_transaction_data.csv`.

It compares non-robust vs robust aggregation under Byzantine attacks (`IPM`, `ALIE`) on a binary fraud-classification task.


In [1]:
import os
import sys
import random
from dataclasses import dataclass
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix

byzfl_src = os.path.abspath("byzantine-fl")
if byzfl_src not in sys.path:
    sys.path.insert(0, byzfl_src)
import byzfl

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
sns.set_theme(style="whitegrid")
device = torch.device("cpu")
device


device(type='cpu')

In [2]:
df = pd.read_csv("bank_transaction_data.csv")
if len(df) > 50000:
    df = df.sample(50000, random_state=SEED).reset_index(drop=True)

df["Fraud Flag"] = df["Fraud Flag"].astype(str).str.lower().map({"true": 1, "false": 0})
df["Transaction Amount"] = pd.to_numeric(df["Transaction Amount"], errors="coerce")
df["Latency (ms)"] = pd.to_numeric(df["Latency (ms)"], errors="coerce")
df["Slice Bandwidth (Mbps)"] = pd.to_numeric(df["Slice Bandwidth (Mbps)"], errors="coerce")
df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")
df = df.dropna(subset=[
    "Sender Account ID", "Receiver Account ID", "Transaction Amount", "Fraud Flag", "Timestamp"
]).copy()

df["hour"] = df["Timestamp"].dt.hour
df["dayofweek"] = df["Timestamp"].dt.dayofweek

cat_cols = ["Transaction Type", "Transaction Status", "Device Used", "Network Slice ID"]
for c in cat_cols:
    df[c] = df[c].astype(str)

X_num = df[["Transaction Amount", "Latency (ms)", "Slice Bandwidth (Mbps)", "hour", "dayofweek"]].copy()
X_num = (X_num - X_num.mean()) / (X_num.std() + 1e-6)

X_cat = pd.get_dummies(df[cat_cols], prefix=cat_cols)
X = pd.concat([X_num, X_cat], axis=1).values.astype(np.float32)
y = df["Fraud Flag"].values.astype(np.int64)

print(f"Rows: {len(X):,}")
print(f"Feature dim: {X.shape[1]}")
print(f"Fraud rate: {y.mean():.4f}")


Rows: 1,000
Feature dim: 15
Fraud rate: 0.4810


In [3]:
# Train/test split and non-IID client partition.
n = len(X)
idx = np.arange(n)
np.random.shuffle(idx)
split = int(0.8 * n)
tr_idx, te_idx = idx[:split], idx[split:]
X_train, y_train = X[tr_idx], y[tr_idx]
X_test, y_test = X[te_idx], y[te_idx]

test_loader = DataLoader(TensorDataset(torch.tensor(X_test), torch.tensor(y_test)), batch_size=1024, shuffle=False)

def make_client_loaders(X_in, y_in, n_clients=8, batch_size=256):
    order = np.argsort(y_in)
    X_in, y_in = X_in[order], y_in[order]
    shards_X = np.array_split(X_in, n_clients)
    shards_y = np.array_split(y_in, n_clients)
    loaders = []
    for Xi, yi in zip(shards_X, shards_y):
        if len(Xi) == 0:
            continue
        ds = TensorDataset(torch.tensor(Xi), torch.tensor(yi))
        # Ensure each client has at least one batch.
        local_bs = max(1, min(batch_size, len(ds)))
        loaders.append(DataLoader(ds, batch_size=local_bs, shuffle=True, drop_last=False))
    return loaders

client_loaders = make_client_loaders(X_train, y_train, n_clients=8, batch_size=256)
len(client_loaders)


8

In [4]:
class LogisticHead(nn.Module):
    def __init__(self, in_dim, out_dim=2):
        super().__init__()
        self.fc = nn.Linear(in_dim, out_dim)
    def forward(self, x):
        return self.fc(x)

def flatten_grads(model):
    return torch.cat([p.grad.detach().clone().view(-1) for p in model.parameters()])

def set_model_grad_from_flat(model, flat_grad):
    start = 0
    for p in model.parameters():
        n = p.numel()
        p.grad = flat_grad[start:start+n].view_as(p).clone()
        start += n

def evaluate(model, loader, return_predictions=False):
    model.eval()
    total, correct, loss_sum = 0, 0, 0.0
    criterion = nn.CrossEntropyLoss()
    ys, ps = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            pred = logits.argmax(dim=1)
            total += yb.numel()
            correct += (pred == yb).sum().item()
            loss_sum += loss.item() * yb.numel()
            if return_predictions:
                ys.append(yb.cpu().numpy())
                ps.append(pred.cpu().numpy())
    avg_loss = loss_sum / total
    avg_acc = correct / total
    if return_predictions:
        return avg_loss, avg_acc, np.concatenate(ys), np.concatenate(ps)
    return avg_loss, avg_acc

@dataclass
class ExperimentConfig:
    name: str
    attack_name: str
    aggregator_name: str
    aggregator: object
    attack: Optional[object]
    f_byz: int

def run_federated_experiment(config, rounds=60, lr=0.2):
    model = LogisticHead(X_train.shape[1]).to(device)
    criterion = nn.CrossEntropyLoss()
    iters = [iter(dl) for dl in client_loaders]
    hist = {"round": [], "test_loss": [], "test_acc": [], "agg_update_norm": []}

    for rnd in range(rounds):
        honest = []
        for i, dl in enumerate(client_loaders):
            try:
                xb, yb = next(iters[i])
            except StopIteration:
                iters[i] = iter(dl)
                xb, yb = next(iters[i])
            xb, yb = xb.to(device), yb.to(device)
            model.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            honest.append(flatten_grads(model))

        honest_stack = torch.stack(honest)
        if config.f_byz > 0 and config.attack is not None:
            byz_vec = config.attack(honest_stack)
            byz_stack = byz_vec.unsqueeze(0).repeat(config.f_byz, 1)
            all_vecs = torch.cat([honest_stack, byz_stack], dim=0)
        else:
            all_vecs = honest_stack

        agg = config.aggregator(all_vecs)
        model.zero_grad()
        set_model_grad_from_flat(model, agg)
        with torch.no_grad():
            for p in model.parameters():
                p -= lr * p.grad

        t_loss, t_acc = evaluate(model, test_loader)
        hist["round"].append(rnd)
        hist["test_loss"].append(t_loss)
        hist["test_acc"].append(t_acc)
        hist["agg_update_norm"].append(torch.norm(agg).item())

    m_loss, m_acc, y_true, y_pred = evaluate(model, test_loader, return_predictions=True)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    metrics = {
        "experiment": config.name,
        "attack": config.attack_name,
        "aggregator": config.aggregator_name,
        "test_loss": m_loss,
        "test_acc": m_acc,
        "precision": p,
        "recall": r,
        "f1": f1,
        "confusion_matrix": confusion_matrix(y_true, y_pred),
    }
    return pd.DataFrame(hist), metrics


In [6]:
f_byz = 2
ipm = byzfl.InnerProductManipulation(tau=3.0)
alie = byzfl.ALittleIsEnough(tau=1.5)

experiments = [
    ExperimentConfig("No attack + Average", "None", "Average", byzfl.Average(), None, 0),
    ExperimentConfig("IPM + Average", "IPM", "Average", byzfl.Average(), ipm, f_byz),
    ExperimentConfig("IPM + Median", "IPM", "Median", byzfl.Median(), ipm, f_byz),
    ExperimentConfig("IPM + TrMean", "IPM", "TrMean", byzfl.TrMean(f=f_byz), ipm, f_byz),
    ExperimentConfig("ALIE + Average", "ALIE", "Average", byzfl.Average(), alie, f_byz),
    ExperimentConfig("ALIE + Median", "ALIE", "Median", byzfl.Median(), alie, f_byz),
    ExperimentConfig("ALIE + TrMean", "ALIE", "TrMean", byzfl.TrMean(f=f_byz), alie, f_byz),
]

all_hist, all_metrics = [], []
for cfg in experiments:
    print("Running:", cfg.name)
    h, m = run_federated_experiment(cfg, rounds=60, lr=0.2)
    h["experiment"] = cfg.name
    h["attack"] = cfg.attack_name
    all_hist.append(h)
    all_metrics.append(m)

results_df = pd.concat(all_hist, ignore_index=True)
metrics_df = pd.DataFrame(all_metrics)
metrics_df[["experiment", "test_acc", "precision", "recall", "f1"]]


Running: No attack + Average


StopIteration: 

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4), sharey=True)
for ax, a in zip(axes, ["None", "IPM", "ALIE"]):
    sub = results_df[results_df["attack"] == a]
    for name, g in sub.groupby("experiment"):
        ax.plot(g["round"], g["test_acc"], label=name)
    ax.set_title(f"Attack: {a}")
    ax.set_xlabel("Round")
    if a == "None":
        ax.set_ylabel("Test accuracy")
    ax.legend(fontsize=8)
plt.suptitle("Accuracy trajectories by attack")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(18, 4), sharey=True)
for ax, metric in zip(axes, ["precision", "recall", "f1"]):
    sns.barplot(data=metrics_df, x=metric, y="experiment", hue="attack", orient="h", ax=ax)
    ax.set_xlim(0, 1)
    ax.set_title(metric.capitalize())
    if metric != "f1":
        ax.legend_.remove()
axes[-1].legend(title="Attack", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.suptitle("Final classification metrics")
plt.tight_layout()
plt.show()


In [ ]:
attacked = metrics_df[metrics_df["attack"] != "None"].reset_index(drop=True)
cols = 3
rows = int(np.ceil(len(attacked) / cols))
fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
axes = np.array(axes).reshape(-1)

for i, (_, row) in enumerate(attacked.iterrows()):
    sns.heatmap(row["confusion_matrix"], annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[i])
    axes[i].set_title(row["experiment"])
    axes[i].set_xlabel("Predicted")
    axes[i].set_ylabel("True")

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.suptitle("Confusion matrices (attacked scenarios)")
plt.tight_layout()
plt.show()


## Notes

- Increase `rounds`, vary `f_byz`, and tune `tau` to stress-test robustness.
- `Median` and `TrMean` generally provide better stability than `Average` under attack.
- Track `f1` and confusion matrices in addition to accuracy for fraud-sensitive evaluation.
